<a href="https://colab.research.google.com/github/SanjaraT/Langchain/blob/main/tool_calling_calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain-ollama

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# Define tools

In [4]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract two numbers."""
    return a - b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@tool
def divide(a: float, b: float) -> float:
    """Divide two numbers."""
    return a / b

In [5]:
tools = [add, subtract, multiply, divide]

In [8]:
# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
# Install zstd dependency
!sudo apt-get install zstd -y


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,458 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently

In [9]:
# Start Ollama server in the background
import subprocess
import time

# Set the OLLAMA_HOST environment variable
import os
os.environ['OLLAMA_HOST'] = '0.0.0.0'

# Start Ollama server in the background
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server some time to start
print("Waiting for Ollama server to start...")
time.sleep(10)
print("Ollama server started.")

Waiting for Ollama server to start...
Ollama server started.


In [10]:
!ollama pull llama3.1:8b

In [11]:
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)


In [12]:
# Bind tools to model
llm_with_tools = llm.bind_tools(tools)

In [13]:
query = "What is 125 multiplied by 67?"


In [14]:
response = llm_with_tools.invoke(query)

print("Content:")
print(response.content)

print("\nTool Calls:")
print(response.tool_calls)

Content:


Tool Calls:
[{'name': 'multiply', 'args': {'a': 125, 'b': 67}, 'id': '3b90a523-e24c-4dc9-b9d6-36c3b0b17506', 'type': 'tool_call'}]


# Tool Execution

In [15]:
if response.tool_calls:

    tool_call = response.tool_calls[0]

    tool_name = tool_call["name"]

    tool_args = tool_call["args"]

    print("\nTool Selected:", tool_name)
    print("Arguments:", tool_args)

    tool_map = {
        "add": add,
        "subtract": subtract,
        "multiply": multiply,
        "divide": divide
    }

    result = tool_map[tool_name].invoke(tool_args)

    print("\nTool Result:")
    print(result)

else:
    print(response.content)


Tool Selected: multiply
Arguments: {'a': 125, 'b': 67}

Tool Result:
8375.0
